In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os 

os.chdir('../..')

BASE_DIR = os.getcwd()

In [3]:
alley_path = BASE_DIR + '/data/raw/amenities/alley_house.csv'
apart_path = BASE_DIR + '/data/raw/amenities/apartment.csv'
land_path = BASE_DIR + '/data/raw/amenities/land.csv'
street_path = BASE_DIR + '/data/raw/amenities/street_house.csv'

In [4]:
category_map = {
    'bệnh viện': 'hospital',
    'chợ': 'market',
    'công viên': 'park',
    'siêu thị': 'supermarket',
    'trung tâm thương mại': 'mall',
    'trường học': 'school'
}

In [5]:
def process_amenities_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    df['Loai_tien_ich_clean'] = df['Loai_tien_ich'].str.strip().str.lower().map(category_map)
    
    df_min = df.groupby(['Ma_BDS', 'Loai_tien_ich_clean'])['Khoảng cách'].min().unstack()
    df_min.columns = [f'dist_min_{col}' for col in df_min.columns]
    
    df_median = df.groupby(['Ma_BDS', 'Loai_tien_ich_clean'])['Khoảng cách'].median().unstack()
    df_median.columns = [f'dist_median_{col}' for col in df_median.columns]
    
    df_1km = df[df['Khoảng cách'] <= 1000]
    df_count_1km = df_1km.groupby(['Ma_BDS', 'Loai_tien_ich_clean']).size().unstack(fill_value=0)
    df_count_1km.columns = [f'count_{col}_1km' for col in df_count_1km.columns]
    
    df_2km = df[df['Khoảng cách'] <= 2000]
    df_count_2km = df_2km.groupby(['Ma_BDS', 'Loai_tien_ich_clean']).size().unstack(fill_value=0)
    df_count_2km.columns = [f'count_{col}_2km' for col in df_count_2km.columns]
    
    df_features = pd.DataFrame(index=df['Ma_BDS'].unique())
    df_features = df_features.join([df_min, df_median, df_count_1km, df_count_2km], how='left')
    
    return df_features

list_amenity_dfs = []
for path in (alley_path, apart_path, street_path, land_path):
    if os.path.exists(path):
        df_processed = process_amenities_file(path)
        list_amenity_dfs.append(df_processed)



In [6]:
df_amenities_all = pd.concat(list_amenity_dfs, axis=0)
df_amenities_all.reset_index(inplace=True)
df_amenities_all.rename(columns={'index': 'Ma_BDS'}, inplace=True)

count_cols = [c for c in df_amenities_all.columns if 'count_' in c]
df_amenities_all[count_cols] = df_amenities_all[count_cols].fillna(0)

dist_cols = [c for c in df_amenities_all.columns if 'dist_' in c]
df_amenities_all[dist_cols] = df_amenities_all[dist_cols].fillna(10000.0)

print(f"{len(df_amenities_all)}")
print("features:")
print(df_amenities_all.columns.tolist())

5256
features:
['Ma_BDS', 'dist_min_hospital', 'dist_min_mall', 'dist_min_market', 'dist_min_park', 'dist_min_school', 'dist_min_supermarket', 'dist_median_hospital', 'dist_median_mall', 'dist_median_market', 'dist_median_park', 'dist_median_school', 'dist_median_supermarket', 'count_hospital_1km', 'count_mall_1km', 'count_market_1km', 'count_park_1km', 'count_school_1km', 'count_supermarket_1km', 'count_hospital_2km', 'count_mall_2km', 'count_market_2km', 'count_park_2km', 'count_school_2km', 'count_supermarket_2km']


In [8]:
df_amenities_all.to_csv(BASE_DIR + '/data/process/merged_amenities_data.csv', index=False)